In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# 1. Enhanced Data Loading and Preparation
def load_and_preprocess(file_path):
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Successfully loaded {len(df)} recipes")
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

    # Clean and preprocess names
    def clean_name(text):
        text = str(text).lower()
        return re.sub(r'[^\w\s]', '', text).strip()
    
    # Clean and preprocess ingredients
    def clean_ingredients(text):
        text = str(text).lower()
        ingredients = re.split(r',|\s+and\s+', text)
        cleaned = []
        for ing in ingredients:
            ing = re.sub(r'\d+[^\s]*', '', ing).strip()  # Remove quantities
            ing = re.sub(r'\s+', '_', ing)  # Replace spaces with underscores
            if ing: cleaned.append(ing)
        return cleaned
    
    df['cleaned_name'] = df['name'].apply(clean_name)
    df['cleaned_ingredients'] = df['ingredients_name'].apply(clean_ingredients)
    
    # Create separate text representations
    df['name_text'] = df['cleaned_name']
    df['ingredient_text'] = df['cleaned_ingredients'].apply(' '.join)
    
    return df

# 2. Enhanced SVD Model
class SVDRecommender:
    def __init__(self, n_components=50):
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.terms = []
        self.term_indices = {}
        
    def create_term_matrix(self, texts):
        # Build vocabulary
        self.terms = []
        for text in texts:
            self.terms.extend(text.split())
        self.terms = list(set(self.terms))
        self.term_indices = {term: idx for idx, term in enumerate(self.terms)}
        
        # Create matrix
        matrix = np.zeros((len(texts), len(self.terms)))
        for i, text in enumerate(texts):
            for term in text.split():
                if term in self.term_indices:
                    matrix[i, self.term_indices[term]] = 1
        return matrix
    
    def fit(self, texts):
        term_matrix = self.create_term_matrix(texts)
        self.svd.fit(term_matrix)
        return self
    
    def transform(self, text):
        vec = np.zeros((1, len(self.terms)))
        for term in text.split():
            if term in self.term_indices:
                vec[0, self.term_indices[term]] = 1
        return self.svd.transform(vec)
    
    def recommend(self, query, dataset, top_n=5):
        query_vec = self.transform(query)
        doc_vectors = self.svd.transform(self.create_term_matrix(dataset))
        similarities = cosine_similarity(query_vec, doc_vectors)[0]
        indices = np.argsort(similarities)[-top_n:][::-1]
        return dataset.iloc[indices], similarities[indices]

# 3. Main Program with Search Mode Selection
if __name__ == "__main__":
    # Load and preprocess data
    df = load_and_preprocess('/kaggle/input/food-recipes/Food_Recipe.csv')
    if df is None:
        exit()

    # Initialize models
    name_model = SVDRecommender(n_components=50)
    ingredient_model = SVDRecommender(n_components=100)
    
    # Train models
    print("\n⚙ Training name model...")
    name_model.fit(df['name_text'])
    print("⚙ Training ingredient model...")
    ingredient_model.fit(df['ingredient_text'])

    # User interaction
    print("\n🔍 Search Options:")
    print("1. Search by Recipe Name")
    print("2. Search by Ingredients")
    choice = input("Choose search mode (1/2): ").strip()

    if choice == '1':
        query = input("Enter recipe name: ").strip().lower()
        query = re.sub(r'[^\w\s]', '', query)  # Clean name query
        model, data = name_model, df['name_text']
    elif choice == '2':
        query = input("Enter ingredients (comma-separated): ").strip().lower()
        # Clean ingredient query
        ingredients = [re.sub(r'\s+', '_', re.sub(r'\d+[^\s]*', '', ing.strip())) 
                      for ing in query.split(',')]
        query = ' '.join(ingredients)
        model, data = ingredient_model, df['ingredient_text']
    else:
        print("❌ Invalid choice!")
        exit()

    # Get recommendations
    results, scores = model.recommend(query, data)
    
    # Display results
    if not results.empty:
        print("\n🍳 Top Recommendations:")
        for i, idx in enumerate(results.index):
            # Convert underscores back to spaces in ingredients
            ingredients = df.loc[idx, 'ingredients_name'].split(', ')  # Split ingredients directly
            print(f"{i+1}. {df.loc[idx, 'name']}")
            print(f"   Ingredients: {', '.join(ingredients[:3])}...")
            print(f"   Match Score: {scores[i]:.2f}\n")
    else:
        print("😞 No matching recipes found")


✅ Successfully loaded 7101 recipes

⚙ Training name model...
⚙ Training ingredient model...

🔍 Search Options:
1. Search by Recipe Name
2. Search by Ingredients


Choose search mode (1/2):  2
Enter ingredients (comma-separated):  chicken,eggs



🍳 Top Recommendations:
1. Mughlai Style Chicken Changezi Recipe
   Ingredients: Curd (Dahi / Yogurt), Ginger Garlic Paste, Coriander Powder (Dhania)...
   Match Score: 0.42

2. Easy Creamy Chicken Curry Recipe
   Ingredients: Chicken, Curd (Dahi / Yogurt), Lemon juice...
   Match Score: 0.36

3. Chicken Malai Kebab Recipe
   Ingredients: Chicken breasts, Onion, Ginger Garlic Paste...
   Match Score: 0.36

4. Dum Ka Murgh (Lagan Ka Murgh) Recipe
   Ingredients: Chicken, Onions, Ginger...
   Match Score: 0.35

5. Dum Murgh Aatishi Recipe - Spicy Smoked Chicken Recipe
   Ingredients: Chicken, Anardana Powder (Pomegranate Seed Powder), Red Chilli powder...
   Match Score: 0.35

